# PFE proposal B — Production RAG assistant: hybrid retrieval + evaluation bench + service
Public stand-in corpus: SQuAD v2.0 dev (Wikipedia paragraphs, CC BY-SA 4.0). In the PFE, swap in the host's manuals.
CPU path: BM25, LSA "dense" stand-in, RRF hybrid, extractive answers, metrics. **[GPU — Colab T4]** cells: BGE-M3 embeddings + cross-encoder reranking.

In [1]:
# Setup (Colab or local). CPU is enough unless a cell says [GPU — Colab T4].
%pip install -q scikit-learn pandas numpy rank_bm25 fastapi httpx

Note: you may need to restart the kernel to use updated packages.


## Package source (embedded from `code/src/`, identical to the tested files)

In [2]:
import os, sys
os.makedirs('src', exist_ok=True)
os.makedirs('src/ragkit', exist_ok=True)
sys.path.insert(0, os.path.abspath('src'))

In [3]:
%%writefile src/ragkit/__init__.py
"""RAG starter kit: BM25 + dense + RRF hybrid retrieval, evaluation bench, extractive/LLM answerers, FastAPI service."""
from . import data, evaluate, generate, retrieval  # noqa: F401


Writing src/ragkit/__init__.py


In [4]:
%%writefile src/ragkit/data.py
"""Public stand-in corpus for development: SQuAD v2.0 dev set (Wikipedia paragraphs + questions).

In the PFE, replace it with the host's technical documentation; keep the same Passage/Query schema.
SQuAD: Rajpurkar et al. — https://rajpurkar.github.io/SQuAD-explorer/ (CC BY-SA 4.0).
"""
from __future__ import annotations

import json
import re
import urllib.request
from dataclasses import dataclass
from pathlib import Path

SQUAD_DEV_URL = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json"


@dataclass(frozen=True)
class Passage:
    pid: str
    title: str
    text: str


@dataclass(frozen=True)
class Query:
    qid: str
    question: str
    gold_pid: str
    answers: tuple[str, ...]


def download_squad(dest: str | Path = "data/squad_dev_v2.json", url: str = SQUAD_DEV_URL) -> Path:
    dest = Path(dest)
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(url, timeout=120) as r:
            dest.write_bytes(r.read())
    return dest


def load_squad(path: str | Path, max_articles: int | None = None) -> tuple[list[Passage], list[Query]]:
    """Each paragraph → Passage; each answerable question → Query whose gold is its paragraph."""
    raw = json.loads(Path(path).read_text(encoding="utf-8"))["data"]
    passages, queries = [], []
    for a_i, art in enumerate(raw[:max_articles] if max_articles else raw):
        for p_i, para in enumerate(art["paragraphs"]):
            pid = f"{a_i}-{p_i}"
            passages.append(Passage(pid, art["title"].replace("_", " "), para["context"]))
            for qa in para["qas"]:
                if qa.get("is_impossible"):
                    continue
                answers = tuple(dict.fromkeys(a["text"] for a in qa["answers"]))
                queries.append(Query(qa["id"], qa["question"], pid, answers))
    return passages, queries


def chunk_text(text: str, max_words: int = 180, overlap: int = 40) -> list[str]:
    """Word-window chunker with overlap, sentence-aligned when possible (for long manuals)."""
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, cur = [], []
    for s in sents:
        w = s.split()
        if cur and len(cur) + len(w) > max_words:
            chunks.append(" ".join(cur))
            cur = cur[-overlap:] if overlap else []
        cur.extend(w)
    if cur:
        chunks.append(" ".join(cur))
    return chunks


Writing src/ragkit/data.py


In [5]:
%%writefile src/ragkit/evaluate.py
"""Retrieval metrics (hit@k, MRR@k), answer metrics (SQuAD EM/F1) and a cheap groundedness proxy."""
from __future__ import annotations

import re
import string
import time
from collections import Counter
from typing import Sequence

import numpy as np

from .data import Query


def hit_at_k(ranked: Sequence[str], gold: str, k: int) -> float:
    return float(gold in list(ranked)[:k])


def reciprocal_rank(ranked: Sequence[str], gold: str, k: int = 10) -> float:
    for i, pid in enumerate(list(ranked)[:k], start=1):
        if pid == gold:
            return 1.0 / i
    return 0.0


def evaluate_retriever(retriever, queries: Sequence[Query], ks=(1, 3, 5, 10)) -> dict:
    kmax = max(ks)
    hits = {k: [] for k in ks}
    rr, lat = [], []
    for q in queries:
        t0 = time.perf_counter()
        ranked = [pid for pid, _ in retriever.search(q.question, kmax)]
        lat.append(time.perf_counter() - t0)
        for k in ks:
            hits[k].append(hit_at_k(ranked, q.gold_pid, k))
        rr.append(reciprocal_rank(ranked, q.gold_pid, kmax))
    out = {f"hit@{k}": float(np.mean(v)) for k, v in hits.items()}
    out[f"mrr@{kmax}"] = float(np.mean(rr))
    out["p95_latency_ms"] = float(np.percentile(lat, 95) * 1000)
    out["n_queries"] = len(queries)
    return out


def _norm(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def exact_match(pred: str, golds: Sequence[str]) -> float:
    return float(any(_norm(pred) == _norm(g) for g in golds))


def f1(pred: str, golds: Sequence[str]) -> float:
    best = 0.0
    p = _norm(pred).split()
    for g in golds:
        gt = _norm(g).split()
        common = Counter(p) & Counter(gt)
        same = sum(common.values())
        if same == 0:
            continue
        prec, rec = same / len(p), same / len(gt)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best


def groundedness(answer: str, contexts: Sequence[str]) -> float:
    """Share of answer content tokens found in the retrieved contexts (proxy; use an LLM judge in the PFE)."""
    a = [t for t in _norm(answer).split() if len(t) > 2]
    if not a:
        return 1.0
    ctx = set(_norm(" ".join(contexts)).split())
    return sum(t in ctx for t in a) / len(a)


Writing src/ragkit/evaluate.py


In [6]:
%%writefile src/ragkit/generate.py
"""Answer generation. Default = extractive baseline (no LLM, CPU, deterministic).
Plug any LLM by passing a callable `llm(prompt: str) -> str` to LLMAnswerer."""
from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Callable, Sequence

from .retrieval import tokenize

PROMPT = """You are a maintenance assistant. Answer ONLY from the context passages below.
Cite the passage ids you used in square brackets, e.g. [3-1]. If the answer is not in the context, reply exactly: "I don't know."

Context:
{context}

Question: {question}
Answer:"""


@dataclass
class Answer:
    text: str
    cited: list[str] = field(default_factory=list)


class ExtractiveAnswerer:
    """Returns the context sentence with the highest token overlap with the question."""

    def __call__(self, question: str, contexts: Sequence[tuple[str, str]]) -> Answer:
        q = set(tokenize(question))
        best, best_pid, best_s = "", None, -1.0
        for pid, text in contexts:
            for sent in re.split(r"(?<=[.!?])\s+", text):
                toks = tokenize(sent)
                if not toks:
                    continue
                s = len(q & set(toks)) / (len(q) ** 0.5 * len(toks) ** 0.5 + 1e-9)
                if s > best_s:
                    best, best_pid, best_s = sent, pid, s
        return Answer(best or "I don't know.", [best_pid] if best_pid else [])


class LLMAnswerer:
    def __init__(self, llm: Callable[[str], str]):
        self.llm = llm

    def __call__(self, question: str, contexts: Sequence[tuple[str, str]]) -> Answer:
        ctx = "\n\n".join(f"[{pid}] {text}" for pid, text in contexts)
        out = self.llm(PROMPT.format(context=ctx, question=question)).strip()
        cited = sorted(set(re.findall(r"\[([^\]]+)\]", out)) & {pid for pid, _ in contexts})
        return Answer(out, cited)


class RAGPipeline:
    def __init__(self, retriever, passages, answerer=None, k: int = 5):
        self.retriever, self.k = retriever, k
        self.by_id = {p.pid: p for p in passages}
        self.answerer = answerer or ExtractiveAnswerer()

    def ask(self, question: str) -> dict:
        hits = self.retriever.search(question, self.k)
        ctx = [(pid, self.by_id[pid].text) for pid, _ in hits]
        ans = self.answerer(question, ctx)
        return {"answer": ans.text, "cited": ans.cited, "retrieved": [pid for pid, _ in hits]}


Writing src/ragkit/generate.py


In [7]:
%%writefile src/ragkit/retrieval.py
"""Sparse (BM25), dense (pluggable encoder) and hybrid retrieval fused with Reciprocal Rank Fusion.

BM25: Robertson & Zaragoza 2009, doi:10.1561/1500000019 · RRF: Cormack et al. 2009, doi:10.1145/1571941.1572114
"""
from __future__ import annotations

import re
import unicodedata
from typing import Protocol, Sequence

import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

from .data import Passage

_STOP = set("a an the of to in on for and or is are was were be by with as at from that this it its which who what when where how why".split())


def tokenize(text: str) -> list[str]:
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return [t for t in re.findall(r"[a-z0-9]+", text) if t not in _STOP]


class Retriever(Protocol):
    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]: ...


class BM25Retriever:
    def __init__(self, passages: Sequence[Passage], k1: float = 1.5, b: float = 0.75):
        self.ids = [p.pid for p in passages]
        self.bm25 = BM25Okapi([tokenize(f"{p.title} {p.text}") for p in passages], k1=k1, b=b)

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        s = self.bm25.get_scores(tokenize(query))
        top = np.argsort(-s)[:k]
        return [(self.ids[i], float(s[i])) for i in top]


class Encoder(Protocol):
    def fit(self, texts: list[str]) -> "Encoder": ...
    def encode(self, texts: list[str]) -> np.ndarray: ...


class LsaEncoder:
    """CPU stand-in for a neural embedder: TF-IDF (word + char n-grams) → truncated SVD, L2-normalised."""

    def __init__(self, dim: int = 256, seed: int = 0):
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
        self.svd = TruncatedSVD(n_components=dim, random_state=seed)

    def fit(self, texts):
        self.svd.fit(self.vec.fit_transform(texts))
        return self

    def encode(self, texts):
        z = self.svd.transform(self.vec.transform(texts))
        return z / (np.linalg.norm(z, axis=1, keepdims=True) + 1e-12)


class SentenceTransformerEncoder:
    """[GPU recommended] e.g. 'BAAI/bge-m3' (Chen et al. 2024, arXiv:2402.03216). Needs sentence-transformers."""

    def __init__(self, name: str = "BAAI/bge-m3", device: str | None = None, batch_size: int = 32):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(name, device=device)
        self.bs = batch_size

    def fit(self, texts):
        return self

    def encode(self, texts):
        return np.asarray(self.model.encode(texts, batch_size=self.bs, normalize_embeddings=True, show_progress_bar=False))


class DenseRetriever:
    def __init__(self, passages: Sequence[Passage], encoder: Encoder):
        self.ids = [p.pid for p in passages]
        texts = [f"{p.title}. {p.text}" for p in passages]
        self.enc = encoder.fit(texts)
        self.mat = self.enc.encode(texts)

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        s = self.mat @ self.enc.encode([query])[0]
        top = np.argsort(-s)[:k]
        return [(self.ids[i], float(s[i])) for i in top]


def rrf_fuse(rankings: Sequence[Sequence[tuple[str, float]]], k: int = 60, top: int = 10) -> list[tuple[str, float]]:
    """score(d) = Σ_r 1 / (k + rank_r(d)), rank starting at 1."""
    agg: dict[str, float] = {}
    for ranking in rankings:
        for rank, (pid, _) in enumerate(ranking, start=1):
            agg[pid] = agg.get(pid, 0.0) + 1.0 / (k + rank)
    return sorted(agg.items(), key=lambda x: -x[1])[:top]


class HybridRetriever:
    def __init__(self, retrievers: Sequence[Retriever], depth: int = 50, k_rrf: int = 60):
        self.retrievers, self.depth, self.k_rrf = list(retrievers), depth, k_rrf

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        return rrf_fuse([r.search(query, self.depth) for r in self.retrievers], k=self.k_rrf, top=k)


class CrossEncoderReranker:
    """[GPU recommended] Re-score the top `depth` candidates of a base retriever with a cross-encoder
    (Nogueira & Cho 2019, arXiv:1901.04085). e.g. 'BAAI/bge-reranker-v2-m3' (GPU) or
    'cross-encoder/ms-marco-MiniLM-L-6-v2' (CPU smoke test)."""

    def __init__(self, base: Retriever, passages: Sequence[Passage], name: str, depth: int = 30, device: str | None = None):
        from sentence_transformers import CrossEncoder
        self.base, self.depth = base, depth
        self.text = {p.pid: f"{p.title}. {p.text}" for p in passages}
        self.model = CrossEncoder(name, device=device)

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        cands = [pid for pid, _ in self.base.search(query, self.depth)]
        scores = self.model.predict([(query, self.text[pid]) for pid in cands], show_progress_bar=False)
        order = np.argsort(-np.asarray(scores))[:k]
        return [(cands[i], float(scores[i])) for i in order]


Writing src/ragkit/retrieval.py


In [8]:
%%writefile src/ragkit/service.py
"""FastAPI service: POST /ask, GET /health. Logs latency per request (feed it to monitoring)."""
from __future__ import annotations

import logging
import time

from fastapi import FastAPI
from pydantic import BaseModel

log = logging.getLogger("ragkit")


class AskIn(BaseModel):
    question: str


def create_app(pipeline) -> FastAPI:
    app = FastAPI(title="RAG assistant")

    @app.get("/health")
    def health():
        return {"status": "ok"}

    @app.post("/ask")
    def ask(body: AskIn):
        t0 = time.perf_counter()
        out = pipeline.ask(body.question)
        out["latency_ms"] = round((time.perf_counter() - t0) * 1000, 2)
        log.info("ask latency_ms=%s retrieved=%s", out["latency_ms"], out["retrieved"][:3])
        return out

    return app


Writing src/ragkit/service.py


In [9]:
from ragkit import data
from ragkit.retrieval import BM25Retriever, DenseRetriever, HybridRetriever, LsaEncoder
from ragkit.evaluate import evaluate_retriever
import random, pandas as pd
path = data.download_squad('data/squad_dev_v2.json')
passages, queries = data.load_squad(path, max_articles=20)   # CPU-sized sample; raise for full runs
random.Random(0).shuffle(queries); queries = queries[:600]
print(len(passages), 'passages |', len(queries), 'queries')

621 passages | 600 queries


In [10]:
bm25 = BM25Retriever(passages)
lsa = DenseRetriever(passages, LsaEncoder(dim=256))
hyb = HybridRetriever([bm25, lsa], depth=50)
table = pd.DataFrame({name: evaluate_retriever(r, queries) for name, r in
                      [('BM25', bm25), ('LSA (dense stand-in)', lsa), ('Hybrid RRF', hyb)]}).T
table.round(3)

,hit@1,hit@3,hit@5,hit@10,mrr@10,p95_latency_ms,n_queries
BM25,0.807,0.915,0.937,0.965,0.866,0.795,600.0
LSA (dense stand-in),0.687,0.862,0.907,0.943,0.783,18.887,600.0
Hybrid RRF,0.783,0.913,0.942,0.962,0.854,16.839,600.0


In [11]:
from ragkit.generate import RAGPipeline
from ragkit.evaluate import f1, groundedness
pipe = RAGPipeline(hyb, passages, k=5)
sample = queries[:100]
res = [pipe.ask(q.question) for q in sample]
ctx = lambda r: [pipe.by_id[p].text for p in r['retrieved']]
print('answer-sentence F1 vs gold (extractive baseline):', round(sum(f1(r['answer'], q.answers) for r, q in zip(res, sample)) / len(sample), 3))
print('groundedness proxy:', round(sum(groundedness(r['answer'], ctx(r)) for r in res) / len(res), 3))
pipe.ask(queries[0].question) | {'question': queries[0].question, 'gold': queries[0].answers}

answer-sentence F1 vs gold (extractive baseline): 0.211
groundedness proxy: 1.0


{'answer': 'Almost all ctenophores are predators – there are no vegetarians and only one genus that is partly parasitic.',
 'cited': ['11-24'],
 'retrieved': ['11-24', '11-25', '11-1', '11-5', '11-6'],
 'question': 'Are ctenophores predators, vegetarian or parasitic?',
 'gold': ('Almost all ctenophores are predators', 'predators')}

In [12]:
# Serve it (FastAPI). In Colab, test in-process with TestClient:
from fastapi.testclient import TestClient
from ragkit.service import create_app
client = TestClient(create_app(pipe))
client.post('/ask', json={'question': queries[1].question}).json()

{'answer': 'Some remote sensing scientists have proposed using the measurement of the radiance coming from vegetation canopies in those bands to characterize plant health status from a satellite platform.',
 'cited': ['7-23'],
 'retrieved': ['7-23', '10-3', '7-22', '10-19', '10-14'],
 'latency_ms': 122.73}

## [GPU — Colab T4] Neural dense retrieval (BGE-M3) + cross-encoder reranking
On CPU this cell uses small models on 100 queries as a smoke test. On a T4 it uses `BAAI/bge-m3` and `BAAI/bge-reranker-v2-m3`.

In [13]:
%pip install -q sentence-transformers
import torch
from ragkit.retrieval import SentenceTransformerEncoder, CrossEncoderReranker
gpu = torch.cuda.is_available()
EMB = 'BAAI/bge-m3' if gpu else 'sentence-transformers/all-MiniLM-L6-v2'
RER = 'BAAI/bge-reranker-v2-m3' if gpu else 'cross-encoder/ms-marco-MiniLM-L-6-v2'
qs = queries if gpu else queries[:100]
neural = DenseRetriever(passages, SentenceTransformerEncoder(EMB, batch_size=64 if gpu else 16))
hyb2 = HybridRetriever([bm25, neural], depth=50)
rerank = CrossEncoderReranker(hyb2, passages, RER, depth=20)
pd.DataFrame({n: evaluate_retriever(r, qs) for n, r in [(f'Dense {EMB.split("/")[-1]}', neural),
              ('Hybrid BM25+neural', hyb2), ('Hybrid + rerank', rerank)]}).T.round(3)

Note: you may need to restart the kernel to use updated packages.


/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3850.28it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3394.84it/s]

,hit@1,hit@3,hit@5,hit@10,mrr@10,p95_latency_ms,n_queries
Dense all-MiniLM-L6-v2,0.69,0.81,0.86,0.90,0.762,8.757,100.0
Hybrid BM25+neural,0.80,0.90,0.92,0.95,0.858,9.584,100.0
Hybrid + rerank,0.90,0.95,0.96,0.97,0.924,846.404,100.0


## Next steps for the PFE
1. Ingest the host's documents (PDF → text/OCR → `chunk_text`), keep metadata filters (product, version, language).
2. Build a 150–300 question gold set with domain experts; track hit@k/MRR + answer faithfulness (LLM-judge, e.g. RAGAS) per release.
3. Gate every index/model change on the bench (no regression > 1 pt), log latency + feedback, monitor query drift.